# 🎤 RVC v2 Training — Pacaveli Voice Clone

Trains an RVC v2 model for audio-to-audio voice conversion — the same tech used in viral deepfake rap tracks.

**What RVC does**: converts YOUR rap audio → Pacaveli's voice, preserving your flow and rhythm.

| Step | Description |
|------|-------------|
| 1 | Verify GPU (T4 required) |
| 2 | Clone RVC WebUI + install deps |
| 3 | Mount Drive, extract dataset |
| 4 | Preprocess + slice audio |
| 5 | Extract HuBERT features |
| 6 | Train RVC v2 (100 epochs, ~1-2 hrs) |
| 7 | Package `rvc_model.pth` + `rvc_model.index` |
| 8 | Install instructions |

**Runtime**: ~1-2 hrs on Colab T4  
**After training**: click 📥 INSTALL MODEL in app → select `rvc_pacaveli.zip`

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), '❌ No GPU — Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   PyTorch: {torch.__version__}')

In [ ]:
# ── Cell 2: Clone RVC WebUI and install dependencies ────────────
import os
os.chdir('/content')

!git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
os.chdir('/content/rvc')

# Install RVC requirements (no-cache to avoid disk overflow)
!pip install -r requirements.txt --no-cache-dir -q

# Download pretrained base models needed for training
!python tools/download_models.py 2>/dev/null || true

# Fallback: download hubert_base.pt manually if not present
import urllib.request
from pathlib import Path

assets_dir = Path('/content/rvc/assets')
hubert_path = assets_dir / 'hubert' / 'hubert_base.pt'
hubert_path.parent.mkdir(parents=True, exist_ok=True)

if not hubert_path.exists():
    print('Downloading hubert_base.pt…')
    urllib.request.urlretrieve(
        'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
        str(hubert_path)
    )

# Download pretrained v2 model
pretrained_dir = assets_dir / 'pretrained_v2'
pretrained_dir.mkdir(parents=True, exist_ok=True)
for fname in ['f0G40k.pth', 'f0D40k.pth']:
    p = pretrained_dir / fname
    if not p.exists():
        print(f'Downloading {fname}…')
        urllib.request.urlretrieve(
            f'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/{fname}',
            str(p)
        )

print('\n✅ RVC WebUI ready')

In [ ]:
# ── Cell 3: Mount Google Drive and extract dataset ───────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, shutil
from pathlib import Path

MODEL_NAME = 'Pacaveli'   # ← change if your model is named differently

# Find training zip
CANDIDATES = [
    f'/content/drive/MyDrive/Pacaveli_vocals_training.zip',
    f'/content/drive/MyDrive/Pacaveli_colab_training.zip',
    f'/content/drive/MyDrive/pacaveli_training.zip',
]
ZIP_PATH = next((z for z in CANDIDATES if Path(z).exists()), None)
if not ZIP_PATH:
    raise FileNotFoundError(
        'Training zip not found in Google Drive root.\n'
        'Expected one of:\n' + '\n'.join(CANDIDATES)
    )

print(f'Found: {ZIP_PATH}')
EXTRACT = Path('/content/dataset_raw')
EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(str(EXTRACT))

# Find audio dir
for candidate in [
    EXTRACT / 'dataset' / MODEL_NAME,
    EXTRACT / MODEL_NAME,
    EXTRACT,
]:
    clips = list(candidate.glob('*.wav')) + list(candidate.glob('*.mp3')) if candidate.is_dir() else []
    if clips:
        AUDIO_DIR = candidate
        break
else:
    raise FileNotFoundError('No audio files found inside zip.')

print(f'Audio dir: {AUDIO_DIR}  ({len(clips)} files)')

In [ ]:
# ── Cell 4: Preprocess — convert to 40kHz WAV and slice ─────────
import subprocess, sys
from pathlib import Path

os.chdir('/content/rvc')

# RVC dataset dir
DATASET_DIR = Path(f'/content/rvc/dataset/{MODEL_NAME}')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Convert all audio to 40kHz mono WAV (RVC's native format)
src_files = list(AUDIO_DIR.glob('*.wav')) + list(AUDIO_DIR.glob('*.mp3')) + \
            list(AUDIO_DIR.glob('*.flac')) + list(AUDIO_DIR.glob('*.m4a'))

print(f'Converting {len(src_files)} clips to 40kHz mono WAV…')
for src in src_files:
    dst = DATASET_DIR / (src.stem + '.wav')
    subprocess.run(
        ['ffmpeg', '-i', str(src), '-ar', '40000', '-ac', '1', '-y', str(dst)],
        capture_output=True
    )

wav_count = len(list(DATASET_DIR.glob('*.wav')))
print(f'Converted: {wav_count} WAV files')

# RVC preprocess: slice audio into segments
print('\nRunning RVC preprocess (slice to segments)…')
LOG_DIR = Path(f'/content/rvc/logs/{MODEL_NAME}')
LOG_DIR.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [
        sys.executable, 'trainset_preprocess_pipeline_print.py',
        str(DATASET_DIR),
        '40000',   # sample rate
        '2',       # number of CPU processes
        str(LOG_DIR),
        'False',   # no pitch guidance during preprocess
    ],
    capture_output=True, text=True, cwd='/content/rvc'
)
print(result.stdout[-1000:] if result.stdout else '(no output)')
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

print('\n✅ Preprocessing complete')

In [ ]:
# ── Cell 5: Extract features (F0 pitch + HuBERT) ────────────────
import subprocess, sys, os
os.chdir('/content/rvc')

# F0 extraction (pitch curves) — use rmvpe for best quality
print('Extracting F0 pitch curves (rmvpe)…')
r = subprocess.run(
    [
        sys.executable, 'extract_f0_print.py',
        str(LOG_DIR),
        '2',         # num processes
        'rmvpe',     # f0 method: rmvpe = best quality
    ],
    capture_output=True, text=True, cwd='/content/rvc'
)
print(r.stdout[-800:] or '(no output)')
if r.returncode != 0:
    print('STDERR:', r.stderr[-400:])

# HuBERT feature extraction
print('\nExtracting HuBERT features…')
r = subprocess.run(
    [
        sys.executable, 'extract_feature_print.py',
        'cuda',       # device
        '1',          # num GPUs
        '0',          # GPU index
        '0',          # rank
        str(LOG_DIR),
        'v2',         # RVC version
    ],
    capture_output=True, text=True, cwd='/content/rvc'
)
print(r.stdout[-800:] or '(no output)')
if r.returncode != 0:
    print('STDERR:', r.stderr[-400:])

print('\n✅ Feature extraction complete')

In [ ]:
# ── Cell 6: Train RVC v2 ─────────────────────────────────────────
import subprocess, sys, os, json
from pathlib import Path
os.chdir('/content/rvc')

# ── Config ──────────────────────────────────────────────────────
EPOCHS      = 100    # 100 = good quality for 30+ min dataset; increase to 200 for more detail
BATCH_SIZE  = 8      # 8 for T4 16GB; reduce to 4 if OOM errors
SAMPLE_RATE = 40000  # must match preprocess step
SAVE_EVERY  = 25     # save checkpoint every N epochs

# Write training config
config_path = LOG_DIR / 'config.json'
train_config = {
    'train': {
        'log_interval': 200,
        'seed': 1234,
        'epochs': EPOCHS,
        'learning_rate': 0.0001,
        'betas': [0.8, 0.99],
        'eps': 1e-9,
        'batch_size': BATCH_SIZE,
        'fp16_run': True,
        'lr_decay': 0.999875,
        'segment_size': 12800,
        'init_lr_ratio': 1,
        'warmup_epochs': 0,
        'c_mel': 45,
        'c_kl': 1.0,
    }
}

print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, sr={SAMPLE_RATE}')
print(f'Estimated time: ~{EPOCHS // 50}-{EPOCHS // 25} hrs on T4')
print('─' * 50)

result = subprocess.run(
    [
        sys.executable, 'train_nsf_sim_cache_sid_load_pretrain.py',
        '-e', MODEL_NAME,
        '-sr', str(SAMPLE_RATE),
        '-f0', '1',             # use f0 (pitch-guided)
        '-bs', str(BATCH_SIZE),
        '-g', '0',              # GPU 0
        '-te', str(EPOCHS),
        '-se', str(SAVE_EVERY),
        '-pg', 'assets/pretrained_v2/f0G40k.pth',
        '-pd', 'assets/pretrained_v2/f0D40k.pth',
        '-l', '1',              # save latest checkpoint
        '-c', '0',              # cache data in GPU (faster, uses VRAM)
        '-sw', '0',             # no sample weight
        '-v', 'v2',             # RVC version 2
    ],
    cwd='/content/rvc',
    text=True,
)

if result.returncode == 0:
    print('\n✅ Training complete!')
else:
    print(f'\n⚠ Training exited with code {result.returncode}')
    print('(Some errors are normal — check if .pth was saved below)')

# List output files
weights_dir = Path('/content/rvc/weights')
if weights_dir.exists():
    pthl = sorted(weights_dir.glob('*.pth'))
    print(f'\nModel files ({len(pthl)}):')
    for p in pthl:
        print(f'  {p.name}  ({p.stat().st_size/1e6:.0f} MB)')

In [ ]:
# ── Cell 7 (optional): Build .index file for higher quality ──────
# The index enables retrieval-based feature matching — improves voice similarity
import subprocess, sys, os
os.chdir('/content/rvc')

print('Building FAISS index (retrieval feature index)…')
result = subprocess.run(
    [
        sys.executable, 'train_index_alone.py',
        MODEL_NAME,   # experiment name
        'v2',         # version
    ],
    capture_output=True, text=True, cwd='/content/rvc'
)
print(result.stdout[-600:] or '(no output)')
if result.returncode != 0:
    print('Index build failed (non-fatal):', result.stderr[-300:])

# Find index files
from pathlib import Path
log_dir = Path(f'/content/rvc/logs/{MODEL_NAME}')
indices = list(log_dir.glob('*.index'))
print(f'\nIndex files: {[i.name for i in indices]}')

In [ ]:
# ── Cell 8: Package and download ─────────────────────────────────
import zipfile, shutil
from pathlib import Path
from google.colab import files

weights_dir = Path('/content/rvc/weights')
log_dir     = Path(f'/content/rvc/logs/{MODEL_NAME}')

# Find latest .pth
pthl = sorted(weights_dir.glob('*.pth'), key=lambda p: p.stat().st_mtime)
if not pthl:
    # Also check logs dir
    pthl = sorted(log_dir.glob('G_*.pth'), key=lambda p: p.stat().st_mtime)
if not pthl:
    raise FileNotFoundError('No .pth found — training may have failed. Check cell 6 output.')

best_pth = pthl[-1]
print(f'Model: {best_pth.name}  ({best_pth.stat().st_size/1e6:.0f} MB)')

# Find index (optional)
index_files = list(log_dir.glob('added_*.index')) + list(log_dir.glob('*.index'))
best_index  = index_files[-1] if index_files else None
if best_index:
    print(f'Index: {best_index.name}  ({best_index.stat().st_size/1e6:.1f} MB)')

# Package as rvc_pacaveli.zip
zip_out = '/content/rvc_pacaveli.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_pth, 'rvc_model.pth')
    if best_index:
        zf.write(best_index, 'rvc_model.index')

size_mb = Path(zip_out).stat().st_size / 1e6
print(f'\n📦 Packaged: rvc_pacaveli.zip  ({size_mb:.0f} MB)')

# Save to Drive
drive_dest = '/content/drive/MyDrive/rvc_pacaveli.zip'
shutil.copy(zip_out, drive_dest)
print(f'☁️  Saved to Google Drive: rvc_pacaveli.zip')

# Download
print('⬇️  Downloading rvc_pacaveli.zip…')
files.download(zip_out)

## 📥 Installing the RVC Model

After downloading `rvc_pacaveli.zip`:

1. Open **AI Vocals Studio**
2. Go to **🎓 Training** tab → click **📥 INSTALL MODEL**
3. Select the downloaded `rvc_pacaveli.zip`

The app extracts:
```
models/Pacaveli/rvc_model.pth     ← main model
models/Pacaveli/rvc_model.index   ← retrieval index (improves quality)
```

On the next **Audio → Audio** generation, the engine badge shows **🎵 RVC v2** and your rapped audio is converted to Pacaveli's voice with your flow intact.

### Re-training tips
- More epochs (150-200) = more voice similarity, risk of over-training
- More data = better generalization
- Try pitch shift `-2` to `-4` in the app if the output sounds too high
